# SFTQwen: Supervised Fine-Tuning of a Small Language Model for Document Summarization

## Installations and dependancies

In [1]:
!pip install \
  "datasets>=3.4.1,<4.4.0" \
  "trl>=0.18.2,<=0.24.0" \
  "transformers>=4.44.0" \
  "accelerate>=0.33.0" \
  "peft>=0.12.0" \
  "bitsandbytes>=0.43.0" \
  unsloth \
  wandb \
  evaluate \
  rouge-score \
  bert-score \
  nltk \
  tqdm 

import os
os.environ["WANDB_PROJECT"] = "qwen2.5-summarization"


## Imports 

To optimize GPU utilization and speed up training, we import Unsloth, a library designed for efficient fine-tuning of large language models.

In [1]:
import random
import re
import gc
from textwrap import fill
from typing import List, Optional, Any
from pprint import pprint

from unsloth import FastLanguageModel, is_bfloat16_supported, get_chat_template
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from transformers import StoppingCriteria, StoppingCriteriaList
import torch
import nltk
import numpy as np
from datasets import Dataset
from peft import LoftQConfig
from trl import SFTTrainer
from peft import PeftModel
import wandb
import evaluate
from tqdm import tqdm
from pydantic import BaseModel, Field
from bert_score import score

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## GPU Requirements

We verifie that the allocated GPU corresponds to a “small” GPU (with less than 16 GB of VRAM), in line with the project requirement to run on a lightweight Google Colab GPU. 

In [2]:
assert torch.cuda.is_available(), "Non detected GPU."

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
torch.cuda.synchronize()

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

use_bf16 = is_bfloat16_supported()
print("bf16 supported:", use_bf16)

!nvidia-smi

GPU = NVIDIA GeForce RTX 5070 Ti. Max memory = 15.469 GB.
0.0 GB of memory reserved.
bf16 supported: True
Mon Dec 22 19:50:23 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti     On  |   00000000:01:00.0 Off |                  N/A |
|  0%   25C    P1             40W /  300W |     232MiB /  16303MiB |      0%      Default |
|                                 

## Configurations

Centralized configuration using Pydantic to make the code modular, readable, and reproducible.
This structure allows easy adaptation of the model, training, generation, and prompts without changing core logic.

In [3]:
class ModelConfig(BaseModel):
    base_model: str = "unsloth/Qwen2.5-1.5B"
    dataset_path: str = "cnn_dataset.arrow"
    max_seq_length: int = 2048
    load_in_4bit: bool = True
    dtype: Optional[str] = None
    chat_template: str = "qwen-2.5"
    eos_token: str = "<|im_end|>"

class LoRAConfig(BaseModel):
    lora_dir: str = "./model"
    r: int = 64
    lora_alpha: int = 64
    lora_dropout: float = 0.20
    bias: str = "none"

    use_gradient_checkpointing: str = "unsloth"
    random_state: int = 42
    use_rslora: bool = False

    loftq_bits: int = 4
    loftq_iter: int = 1

class TrainingConfig(BaseModel):
    output_dir: str = "/content/qwen_summarizer_lora"
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int = 8
    gradient_accumulation_steps: int = 8

    learning_rate: float = 2e-4
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.03

    logging_strategy: str = "steps"
    logging_steps: int = 1
    eval_strategy: str = "steps"
    eval_steps: int = 10

    save_steps: int = 100
    save_total_limit: int = 3

    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    seed: int = 42

    optim: str = "adamw_8bit"
    report_to: str = "wandb"

class GenerationConfig(BaseModel):
    n_sentences: int = 3
    max_new_tokens: int = 160
    max_input_length: int = 2048

    do_sample: bool = False
    repetition_penalty: float = 1.0
    no_repeat_ngram_size: int = 3

class PromptConfig(BaseModel):
    system_prompt: str = (
        "You are a professional news summarization assistant. "
    )

    user_prompt: str = (
        "Summarize the following news article in at most 3 sentences. "
        "Rewrite the information concisely in your own words. "
        "Focus on the main events and key facts. "
    
    )

class Configuration(BaseModel):
    model: ModelConfig = ModelConfig()
    lora: LoRAConfig = LoRAConfig()
    training: TrainingConfig = TrainingConfig()
    generation: GenerationConfig = GenerationConfig()
    prompt: PromptConfig = PromptConfig()

config = Configuration()

## Model Initialization and Tokenizer Configuration

Ensure the EOS token matches the Qwen chat message terminator (<|im_end|>) to guarantee correct sequence termination.

In [4]:
model_cfg = config.model

print("\n===== Model CONFIG =====")
pprint(model_cfg.model_dump())
print("=========================\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_cfg.base_model,
    max_seq_length = model_cfg.max_seq_length,
    dtype          = model_cfg.dtype,
    load_in_4bit   = model_cfg.load_in_4bit,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template=model_cfg.chat_template,
)

IM_END = model_cfg.eos_token
im_end_id = tokenizer.convert_tokens_to_ids(IM_END)

if tokenizer.eos_token_id is None or tokenizer.eos_token_id != im_end_id:
    tokenizer.eos_token = IM_END
    tokenizer.eos_token_id = im_end_id

print(
    "eos_token:", tokenizer.eos_token,
    "| eos_token_id:", tokenizer.eos_token_id
)


===== Model CONFIG =====
{'base_model': 'unsloth/Qwen2.5-1.5B',
 'chat_template': 'qwen-2.5',
 'dataset_path': 'cnn_dataset.arrow',
 'dtype': None,
 'eos_token': '<|im_end|>',
 'load_in_4bit': True,
 'max_seq_length': 2048}

==((====))==  Unsloth 2025.12.8: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.469 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
eos_token: <|im_end|> | eos_token_id: 151645


In [5]:
lora_cfg = config.lora

print("\n===== LoRA CONFIG =====")
pprint(lora_cfg.model_dump())
print("=========================\n")

loftq_config = LoftQConfig(
    loftq_bits=lora_cfg.loftq_bits,
    loftq_iter=lora_cfg.loftq_iter,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg.r,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_cfg.lora_alpha,
    lora_dropout=lora_cfg.lora_dropout,
    bias=lora_cfg.bias,
    use_gradient_checkpointing=lora_cfg.use_gradient_checkpointing,
    random_state=lora_cfg.random_state,
    use_rslora=lora_cfg.use_rslora,
    loftq_config=loftq_config,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.2.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.



===== LoRA CONFIG =====
{'bias': 'none',
 'loftq_bits': 4,
 'loftq_iter': 1,
 'lora_alpha': 64,
 'lora_dir': './model',
 'lora_dropout': 0.2,
 'r': 64,
 'random_state': 42,
 'use_gradient_checkpointing': 'unsloth',
 'use_rslora': False}



/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/config.py:730: UserWarning: `loftq_config` specified but will be ignored when `init_lora_weights` is not 'loftq'.
  warnings.warn("`loftq_config` specified but will be ignored when `init_lora_weights` is not 'loftq'.")
Unsloth 2025.12.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 73,859,072 || all params: 1,617,573,376 || trainable%: 4.5660


This block applies parameter-efficient fine-tuning (PEFT) using LoRA on top of the frozen base model. LoRA adapters are injected into the main attention and MLP projection layers, drastically reducing the number of trainable parameters. LoftQ is enabled to better initialize LoRA weights when training on a 4-bit quantized model, improving stability and convergence while keeping GPU memory usage low.

## Supervised Fine-Tuning Dataset Preprocessing

This cell builds the supervised fine-tuning dataset in chat format. The prompt (system + user) is masked so that the loss is computed only on the assistant’s summary. Sequences are tokenized, truncated to a fixed length, and split into train, validation, and test sets.

You must run Annotator-notebook.ipynb before to generate the synthetic summaries used for supervised fine-tuning.

In [6]:
prompt_cfg = config.prompt

print("\n===== PROMPT CONFIG =====")
pprint(prompt_cfg.model_dump())
print("===========================\n")

MAX_SEQ_LENGTH = config.model.max_seq_length
EOS_TOKEN = config.model.eos_token
SYSTEM_PROMPT = config.prompt.system_prompt
USER_PROMPT = config.prompt.user_prompt

def process_func(example: dict[str, str]) -> dict[str, list[int]]:
    """
    Tokenize a single training example into model inputs and labels.

    Args:
        example (dict[str, str]): Dataset example containing document and summary.

    Returns:
        dict[str, list[int]]: Tokenized inputs with labels.
    """
    ignore_index = -100
    messages_prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            USER_PROMPT +
            f"ARTICLE:\n{example['document']}"
        )},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages_prompt,
        tokenize=False,
        add_generation_prompt=True,
    )

    answer_text = example["summary"].rstrip() + "\n" + EOS_TOKEN

    prompt_tok = tokenizer(prompt_text, add_special_tokens=False)
    answer_tok = tokenizer(answer_text, add_special_tokens=False)

    input_ids = prompt_tok["input_ids"] + answer_tok["input_ids"]
    attention_mask = prompt_tok["attention_mask"] + answer_tok["attention_mask"]

    labels = [ignore_index] * len(prompt_tok["input_ids"]) + answer_tok["input_ids"]

    input_ids = input_ids[:MAX_SEQ_LENGTH]
    attention_mask = attention_mask[:MAX_SEQ_LENGTH]
    labels = labels[:MAX_SEQ_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


def tokenize(ds: Dataset) -> Dataset:
    """
    Tokenize a dataset for supervised fine-tuning.

    Args:
        ds (Dataset): Input dataset.

    Returns:
        Dataset: Tokenized dataset.
    """
    return ds.map(
        process_func,
        remove_columns=ds.column_names,
        num_proc=2,
    )


raw = Dataset.from_file(model_cfg.dataset_path)

raw_split = raw.train_test_split(test_size=0.10, seed=42)
raw_train_val = raw_split["train"]
raw_test = raw_split["test"]

raw_train_val = raw_train_val.train_test_split(test_size=0.05, seed=42)
raw_train = raw_train_val["train"]
raw_val = raw_train_val["test"]

train_dataset = tokenize(raw_train)
val_dataset   = tokenize(raw_val)
test_dataset  = tokenize(raw_test)

print("Example of input ids: ", train_dataset[0]['input_ids'][:10])
print("Example of attention mask: ", train_dataset[0]['attention_mask'][:10])
print("Example of label ids: ", train_dataset[0]['labels'][-100:])


===== PROMPT CONFIG =====
{'system_prompt': 'You are a professional news summarization assistant. ',
 'user_prompt': 'Summarize the following news article in at most 3 sentences. '
                'Rewrite the information concisely in your own words. Focus on '
                'the main events and key facts. '}



Map (num_proc=2):   0%|          | 0/4275 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/225 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

Example of input ids:  [151644, 8948, 198, 2610, 525, 264, 6584, 3669, 28285, 2022]
Example of attention mask:  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Example of label ids:  [-100, 52, 808, 13, 3321, 13, 70714, 6712, 479, 3092, 2260, 11, 879, 25882, 264, 10441, 304, 6058, 220, 17, 15, 16, 16, 429, 2115, 1059, 448, 264, 1968, 10895, 11, 702, 5880, 458, 7699, 2311, 24849, 330, 74639, 1694, 25, 362, 15106, 315, 98530, 323, 17758, 1, 911, 1059, 13351, 323, 2272, 448, 46633, 9972, 4389, 18661, 13, 576, 2311, 11, 1062, 66283, 448, 41007, 1863, 300, 10303, 11, 3565, 862, 5025, 11, 1059, 4948, 6931, 11, 323, 279, 30826, 315, 279, 10441, 11, 448, 2176, 479, 3092, 2260, 323, 18661, 5290, 19325, 315, 279, 46368, 73328, 624, 151645]


## Supervised Fine-Tuning Configuration and Training

Training configuration : the setup is optimized for small GPUs through gradient accumulation, 8-bit optimization, and mixed-precision training, while monitoring progress with Weights & Biases. Evaluation and checkpointing are performed regularly to track convergence and prevent overfitting.

In [ ]:
train_cfg = config.training

print("\n===== TRAINING CONFIG =====")
pprint(train_cfg.model_dump())
print("==============================\n")

wandb.finish()
wandb.init(
    name=train_cfg.output_dir.split("/")[-1], 
)

training_args = TrainingArguments(
    output_dir=train_cfg.output_dir,

    num_train_epochs=train_cfg.num_train_epochs,
    per_device_train_batch_size=train_cfg.per_device_train_batch_size,
    per_device_eval_batch_size=train_cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=train_cfg.gradient_accumulation_steps,

    learning_rate=train_cfg.learning_rate,
    lr_scheduler_type=train_cfg.lr_scheduler_type,
    warmup_ratio=train_cfg.warmup_ratio,

    logging_strategy="steps",
    logging_steps=train_cfg.logging_steps,

    eval_strategy="steps",
    eval_steps=train_cfg.eval_steps,

    save_strategy="steps",
    save_steps=train_cfg.save_steps,
    save_total_limit=train_cfg.save_total_limit,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to=train_cfg.report_to,

    bf16=use_bf16,
    fp16=not use_bf16,

    optim=train_cfg.optim,
    weight_decay=train_cfg.weight_decay,
    max_grad_norm=train_cfg.max_grad_norm,
    seed=train_cfg.seed,
)


data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    max_seq_length=model_cfg.max_seq_length,
    data_collator=data_collator,
    packing=False,
)


===== TRAINING CONFIG =====
{'eval_steps': 10,
 'eval_strategy': 'steps',
 'gradient_accumulation_steps': 8,
 'learning_rate': 0.0002,
 'logging_steps': 1,
 'logging_strategy': 'steps',
 'lr_scheduler_type': 'cosine',
 'max_grad_norm': 1.0,
 'num_train_epochs': 3,
 'optim': 'adamw_8bit',
 'output_dir': '/content/qwen_summarizer_lora',
 'per_device_eval_batch_size': 8,
 'per_device_train_batch_size': 8,
 'report_to': 'wandb',
 'save_steps': 100,
 'save_total_limit': 3,
 'seed': 42,
 'warmup_ratio': 0.03,
 'weight_decay': 0.01}



wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: marius-dragic (marius-dragic-centralesup-lec) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


In [8]:
from unsloth import unsloth_train
trainer_stats = unsloth_train(trainer)

print(trainer_stats)

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory during training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,275 | Num Epochs = 3 | Total steps = 201
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 73,859,072 of 1,617,573,376 (4.57% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,0.984000,0.956684
20,0.951700,0.921160
30,0.974900,0.902173
40,0.906100,0.888822
50,0.894500,0.879647
60,0.916700,0.874787
70,0.830000,0.872908
80,0.777000,0.880637
90,0.742600,0.877302
100,0.744700,0.876319


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


eval/loss,█▅▄▂▂▁▁▂▁▁▁▁▁▂▅▅▄▄▄▄
eval/runtime,█▃▂▂▃▃▂▃▃▄▄▃▂▂▅▃▄▃▁▃
eval/samples_per_second,▁▆▇▇▆▆▇▆▆▅▅▆▇▇▄▆▅▆█▆
eval/steps_per_second,▁▆▇▇▆▆▇▅▆▅▄▆▇▇▄▆▅▆█▆
train/epoch,▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇██
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇████
train/grad_norm,█▇▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂▂▂▂▂▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂
train/learning_rate,▂▅██████▇▇▇▇▇▇▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
train/loss,█▇▇▆▆▅▆▅▅▅▅▅▅▅▅▄▄▃▃▃▃▃▃▃▃▃▃▃▂▁▁▂▂▁▂▁▁▁▁▁
eval/loss,0.91176
eval/runtime,6.6362


TrainOutput(global_step=201, training_loss=0.7663791888388828, metrics={'train_runtime': 1348.9635, 'train_samples_per_second': 9.507, 'train_steps_per_second': 0.149, 'total_flos': 5.387701589348352e+16, 'train_loss': 0.7663791888388828, 'epoch': 3.0})
1348.9635 seconds used for training.
22.48 minutes used for training.
Peak reserved memory during training = 6.15 GB.
Peak reserved memory % of max memory = 39.757 %.


In [9]:
lora_cfg = config.lora

trainer.model.save_pretrained(lora_cfg.lora_dir)
tokenizer.save_pretrained(lora_cfg.lora_dir)

print("LoRA adapter saved to:", lora_cfg.lora_dir)

LoRA adapter saved to: ./model


## Inference for summary generation 

In [10]:
generation_cfg = config.generation

print("\n===== PROMPT CONFIG =====")
pprint(generation_cfg.model_dump())
print("===========================\n")

def load_model_for_inference(
    base_model_name_or_path: str,
    lora_path: str | None = None,
    max_seq_length: int = 2048,
    load_in_4bit: bool = True,
):
    """
    Load a base model with optional LoRA adapters for optimized inference.

    Args:
        base_model_name_or_path (str): Base model name or path.
        lora_path (str | None): Path to LoRA adapters, if any.
        max_seq_length (int): Maximum sequence length.
        load_in_4bit (bool): Whether to load the model in 4-bit precision.

    Returns:
        tuple: Loaded model and tokenizer.
    """

    torch.set_grad_enabled(False)
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_name_or_path,
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        torch_dtype=None,
        device_map="auto",
    )

    if lora_path is not None:
        model = PeftModel.from_pretrained(
            model,
            lora_path,
            is_trainable=False, 
        )

    FastLanguageModel.for_inference(model)
    model.eval()

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


model, tokenizer = load_model_for_inference(
    base_model_name_or_path=model_cfg.base_model,
    lora_path=lora_cfg.lora_dir,
    max_seq_length=generation_cfg.max_input_length,
)



===== PROMPT CONFIG =====
{'do_sample': False,
 'max_input_length': 2048,
 'max_new_tokens': 160,
 'n_sentences': 3,
 'no_repeat_ngram_size': 3,
 'repetition_penalty': 1.0}

==((====))==  Unsloth 2025.12.8: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.469 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


This custom stopping criterion is used because the model sometimes loops or fails to emit the <|im_end|> token during generation. To ensure concise outputs and prevent runaway generations, decoding is explicitly stopped once three sentences have been produced.

In [11]:
class StopAfterNSentences(StoppingCriteria):
    """
    Stop text generation after a fixed number of sentences.
    """

    def __init__(self, tokenizer, n_sentences: int = 3):
        """
        Args:
            tokenizer (PreTrainedTokenizerBase): Tokenizer used for decoding.
            n_sentences (int): Maximum number of sentences to generate.
        """
        self.tokenizer = tokenizer
        self.n_sentences = n_sentences

        self.sentence_regex = re.compile(
            r"(?<!\b[A-Z])([.!?])(?=\s|$)"
        )

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        """
        Check whether generation should stop.

        Args:
            input_ids (torch.Tensor): Generated token IDs.
            scores (torch.Tensor): Model scores (unused).

        Returns:
            bool: True if generation should stop.
        """
        decoded = self.tokenizer.decode(
            input_ids[0],
            skip_special_tokens=False
        )

        if "<|im_start|>assistant\n" in decoded:
            decoded = decoded.split("<|im_start|>assistant\n", 1)[1]

        sentence_count = len(self.sentence_regex.findall(decoded))
        return sentence_count >= self.n_sentences
        

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

Function performs batched inference by formatting inputs in the Qwen chat style and generating summaries in evaluation mode without gradient computation. Generation is constrained using decoding penalties and a custom stopping criterion to ensure concise, non-repetitive summaries limited to a fixed number of sentences.

In [12]:
def generate_summary(documents):
    """
    Generate summaries for a batch of documents using config-driven generation
    and prompt settings.
    """

    batch_messages = [
        [
            {"role": "system", "content": prompt_cfg.system_prompt},
            {"role": "user", "content": (
                prompt_cfg.user_prompt +
                f"ARTICLE:\n{doc}"
            )},
        ]
        for doc in documents
    ]

    input_ids = tokenizer.apply_chat_template(
        batch_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=generation_cfg.max_input_length,
    ).to("cuda")

    stopping_criteria = StoppingCriteriaList([
        StopAfterNSentences(
            tokenizer,
            n_sentences=generation_cfg.n_sentences,
        ),
    ])

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=generation_cfg.max_new_tokens,
            do_sample=generation_cfg.do_sample,
            repetition_penalty=generation_cfg.repetition_penalty,
            no_repeat_ngram_size=generation_cfg.no_repeat_ngram_size,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria,
            use_cache=True,
        )

    preds = []
    for out in outputs:
        decoded = tokenizer.decode(out, skip_special_tokens=False)
        decoded = decoded.split("<|im_start|>assistant\n", 1)[-1]
        decoded = decoded.split(model_cfg.eos_token, 1)[0]
        preds.append(decoded.strip())

    return preds

## Random summary example

In [13]:
ex = random.choice(raw_test)

article = ex["document"]
gold = ex["summary"]

pred = generate_summary([article])[0]

print("="*110)
print("ARTICLE:\n", fill(article, 110))
print("="*110)
print("GOLD SUMMARY:\n", fill(gold, 110))
print("="*110)
print("MODEL SUMMARY:\n", fill(pred, 110))
print("="*110)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


ARTICLE:
 Charges have been dropped against four men accused of raping an 18-year-old student at Hofstra University
after the woman recanted her allegations, prosecutors said. A Hofstra University student recanted her claims
that she was lured to a dorm and assaulted in a bathroom stall. A judge dismissed all charges Wednesday night
and ordered the release of the four men -- Jesus Ortiz, 19; Stalin Felipe, 19; Kevin Taveras, 20; and Rondell
Bedward, 21; all of the New York metropolitan area, according to Nassau County, New York, District Attorney
Kathleen Rice. They had been arrested, arraigned and jailed, with bail set at $500,000 each. Each was facing
five counts of first-degree rape. "Late this evening, during the continuation of the Nassau County Police
Department's investigation of the allegation, and under questioning by my office's chief trial attorney and
chief sex crimes prosecutor, the alleged victim of the sexual assault admitted that the encounter that took
place early Sund

## Evaluation Strategy

We evaluate the summarization quality using standard, widely adopted metrics. ROUGE measures n-gram overlap, METEOR accounts for synonymy and linguistic variation, and BERTScore captures semantic similarity using contextual embeddings, providing a balanced assessment of lexical and semantic fidelity.

In [14]:
nltk.download("wordnet")
nltk.download("omw-1.4")

rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [15]:
BATCH_SIZE = 4

predictions = []
references = []

docs_buffer = []
refs_buffer = []

for ex in tqdm(raw_test, desc="Evaluating on val (batched)"):
    docs_buffer.append(ex["document"])
    refs_buffer.append(ex["summary"].strip())

    if len(docs_buffer) == BATCH_SIZE:
        preds = generate_summary(docs_buffer)
        predictions.extend(preds)
        references.extend(refs_buffer)

        docs_buffer = []
        refs_buffer = []

if len(docs_buffer) > 0:
    preds = generate_summary(docs_buffer)
    predictions.extend(preds)
    references.extend(refs_buffer)


Evaluating on val (batched): 100%|██████████| 500/500 [08:56<00:00,  1.07s/it]


In [16]:
rouge_scores = rouge.compute(
    predictions=predictions,
    references=references,
)

print("=== ROUGE ===")
for k in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
    print(f"{k}: {rouge_scores[k]:.4f}")


=== ROUGE ===
rouge1: 0.3608
rouge2: 0.1203
rougeL: 0.2421
rougeLsum: 0.2530


In [17]:
meteor_score = meteor.compute(
    predictions=predictions,
    references=references,
)

print("\n=== METEOR ===")
print(f"meteor: {meteor_score['meteor']:.4f}")


=== METEOR ===
meteor: 0.2906


In [18]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

torch.cuda.synchronize()
torch.set_grad_enabled(False)

P, R, F1 = score(
    predictions,
    references,
    lang="en",
    model_type="roberta-base",   
    device="cuda",           
    rescale_with_baseline=True,
    verbose=True,
)

bert_p = P.mean().item()
bert_r = R.mean().item()
bert_f1 = F1.mean().item()

print("\n=== BERTScore (roberta-base, GPU) ===")
print(f"Precision: {bert_p:.4f}")
print(f"Recall:    {bert_r:.4f}")
print(f"F1:        {bert_f1:.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/16 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/8 [00:00<?, ?it/s]

done in 2.63 seconds, 189.80 sentences/sec

=== BERTScore (roberta-base, GPU) ===
Precision: 0.2833
Recall:    0.3332
F1:        0.3068


In [19]:
results = {
    "rouge1": rouge_scores["rouge1"],
    "rouge2": rouge_scores["rouge2"],
    "rougeL": rouge_scores["rougeL"],
    "meteor": meteor_score["meteor"],
    "bertscore_f1": bert_f1,
}

print("\n=== FINAL VALIDATION RESULTS ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")



=== FINAL VALIDATION RESULTS ===
rouge1: 0.3608
rouge2: 0.1203
rougeL: 0.2421
meteor: 0.2906
bertscore_f1: 0.3068


## Conclusion

The fine-tuned Qwen2.5-1.5B model achieves reasonable summarization performance, with a ROUGE-1 score of 0.36 indicating acceptable content coverage under tight length constraints. However, the relatively low ROUGE-2 (0.12) suggests limited modeling of longer phrasal dependencies, which is expected for a small model trained on noisy, automatically generated labels. The METEOR score of 0.29 shows some paraphrasing ability, but also highlights remaining lexical and stylistic inconsistencies. The BERTScore F1 of 0.30 reflects moderate semantic alignment, constrained by both model capacity and teacher-label imperfections. Overall, the results confirm the feasibility of the teacher–student approach, while leaving clear room for improvement through cleaner supervision and stronger distillation strategies.